In [17]:
import pandas as pd
from pathlib import Path

project_root = Path("..").resolve()
manifest_path = project_root / "data" / "processed" / "image_manifest.csv"

manifest = pd.read_csv(manifest_path, dtype={"camera_id": "string"})
timestamp = pd.to_datetime(manifest["timestamp"])

manifest["date"] = timestamp.dt.normalize()
manifest["hour"] = timestamp.dt.hour
manifest["is_weekend"] = timestamp.dt.dayofweek >= 5

coverage = (
    manifest.groupby(["camera_id", "hour", "is_weekend"], as_index=False)
    .agg(n_images=("date", "size"), n_days=("date", "nunique"))
)

coverage.head(10)

C:\Users\NEXT Speed\AppData\Local\Temp\ipykernel_32412\721157966.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  manifest["is_weekend"] = timestamp.dt.dayofweek >= 5


,camera_id,hour,is_weekend,n_images,n_days
0,1028,6,False,22,4
1,1028,6,True,12,2
2,1028,7,False,24,4
3,1028,7,True,11,2
4,1028,8,False,25,5
5,1028,8,True,12,2
6,1028,9,False,30,5
7,1028,9,True,12,2
8,1028,10,False,30,5
9,1028,10,True,12,2


In [18]:
days_by_type = coverage.pivot(
    index=["camera_id", "hour"],
    columns="is_weekend",
    values="n_days",
).reindex(columns=[False, True]).fillna(0)

# count camera weekday and weekend
has_both = days_by_type.gt(0).all(axis=1)
has_both.groupby(level="hour").sum().rename("n_cameras")

hour
6     100
7     100
8     100
9     100
10    100
11    100
12    100
13    100
14    100
15    100
16    100
17    100
18    100
19    100
20    100
21     70
Name: n_cameras, dtype: int64

In [19]:
print(has_both.groupby(level="hour").sum().rename("n_cameras"))

hour
6     100
7     100
8     100
9     100
10    100
11    100
12    100
13    100
14    100
15    100
16    100
17    100
18    100
19    100
20    100
21     70
Name: n_cameras, dtype: int64
